In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
!pip install timm

# **Code Ready**

In [ ]:
!git clone https://github.com/facebookresearch/mae

In [ ]:
%cd mae

# **Data Ready**

In [ ]:
!cp ../gdrive/MyDrive/bracs/train.zip ./train.zip
!unzip train.zip
!rm train.zip
!rm -r ./train/1
!rm -r ./train/2
!rm -r ./train/4
!rm -r ./train/6

In [ ]:
!cp ../gdrive/MyDrive/bracs/bracs_in_test.zip ./bracs_in_test.zip
!unzip bracs_in_test.zip
!rm bracs_in_test.zip
!rm -r ./test/1
!rm -r ./test/2
!rm -r ./test/4
!rm -r ./test/6

In [7]:
%mkdir data
%mv ./train ./data/train
%mv ./test ./data/val

# **Train Vanilla**

<font color="yellow">
download your pretrained model to resume it
</font>

In [8]:
%mkdir model

In [9]:
# this is mae base pretrained model
!wget -O ./model/checkpoint.pth https://dl.fbaipublicfiles.com/mae/finetune/mae_finetuned_vit_base.pth

--2024-01-14 10:30:51--  https://dl.fbaipublicfiles.com/mae/finetune/mae_finetuned_vit_base.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.226.210.25, 13.226.210.78, 13.226.210.111, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.226.210.25|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 346326087 (330M) [binary/octet-stream]
Saving to: ‘./model/checkpoint.pth’

./model/checkpoint. 100%[===================>] 330.28M   145MB/s    in 2.3s    

2024-01-14 10:30:54 (145 MB/s) - ‘./model/checkpoint.pth’ saved [346326087/346326087]



<font color="red">
change main_pretrain.py:
<br>
comment line 27
<br>
line 179 -> param_groups = optim_factory.param_groups_weight_decay(model_without_ddp, args.weight_decay)
<br>
line 197 -> epoch % 1
<br> <br>
change util.misc.py:
<br>
line 21 -> from torch import inf
<br>
line 322 -> add strict=False
<br> <br>
change models_mae.py:
<br>
delete 'qk_scale=None' argument in line 40 and 54
</fon>

In [26]:
!python main_pretrain.py \
    --batch_size 32 \
    --epochs 10 \
    --model mae_vit_base_patch16 \
    --warmup_epochs 1 \
    --data_path ./data \
    --resume ./model/checkpoint.pth \
    --num_workers 2

2024-01-14 11:10:36.297814: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-01-14 11:10:36.297887: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-01-14 11:10:36.300036: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-01-14 11:10:36.312020: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-01-14 11:10:37.863816: W tensorflow/comp

In [27]:
%cp ./output_dir/checkpoint-9.pth ../gdrive/MyDrive/MAE/checkpoint-9.pth